# Statistical Significance Analysis: Alzheimer's Disease Features

This notebook identifies which features are **statistically significant** predictors of Alzheimer's disease diagnosis using formal hypothesis testing.

**Methods used:**
- Independent samples t-tests (continuous features)
- Chi-square tests of independence (categorical/binary features)
- Mann-Whitney U tests (non-parametric alternative)
- Logistic regression with p-values
- Random Forest feature importance
- Bonferroni correction for multiple comparisons

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import ttest_ind, chi2_contingency, mannwhitneyu
import statsmodels.api as sm
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', None)

In [ ]:
# Load dataset
df = pd.read_csv('alzheimers_disease_data.csv')
print(f"Dataset shape: {df.shape}")
print(f"\nDiagnosis distribution:")
print(df['Diagnosis'].value_counts())
print(f"\nDiagnosis rate: {df['Diagnosis'].mean()*100:.1f}%")

In [ ]:
# Drop non-informative columns
df_analysis = df.drop(columns=['PatientID', 'DoctorInCharge'])

# Separate groups
diagnosed = df_analysis[df_analysis['Diagnosis'] == 1]
not_diagnosed = df_analysis[df_analysis['Diagnosis'] == 0]

print(f"Diagnosed (Alzheimer's): {len(diagnosed)} patients")
print(f"Not diagnosed: {len(not_diagnosed)} patients")

## 1. Independent Samples T-Tests (Continuous Features)

Testing whether the mean of each continuous feature differs significantly between diagnosed and non-diagnosed groups.

- **H₀**: No difference in means between groups
- **H₁**: Significant difference in means between groups
- **α = 0.05**

In [ ]:
# Identify continuous features (more than 2 unique values and float-like)
continuous_features = [col for col in df_analysis.columns 
                       if col != 'Diagnosis' and df_analysis[col].nunique() > 10]

# Binary/categorical features
binary_features = [col for col in df_analysis.columns 
                   if col != 'Diagnosis' and df_analysis[col].nunique() <= 10]

print(f"Continuous features ({len(continuous_features)}): {continuous_features}")
print(f"\nBinary/Categorical features ({len(binary_features)}): {binary_features}")

In [ ]:
# Perform t-tests on continuous features
ttest_results = []

for feature in continuous_features:
    group1 = diagnosed[feature].dropna()
    group2 = not_diagnosed[feature].dropna()
    
    t_stat, p_value = ttest_ind(group1, group2)
    
    # Effect size (Cohen's d)
    pooled_std = np.sqrt((group1.std()**2 + group2.std()**2) / 2)
    cohens_d = (group1.mean() - group2.mean()) / pooled_std if pooled_std > 0 else 0
    
    ttest_results.append({
        'Feature': feature,
        'Mean (Diagnosed)': group1.mean(),
        'Mean (Not Diagnosed)': group2.mean(),
        'T-Statistic': t_stat,
        'P-Value': p_value,
        'Cohen\'s d': cohens_d,
        'Significant (p<0.05)': p_value < 0.05
    })

ttest_df = pd.DataFrame(ttest_results).sort_values('P-Value')
print("T-Test Results (sorted by p-value):")
print("=" * 80)
ttest_df

In [ ]:
# Visualize t-test results
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# P-values
significant = ttest_df[ttest_df['Significant (p<0.05)']]
not_significant = ttest_df[~ttest_df['Significant (p<0.05)']]

ax = axes[0]
colors = ['#e74c3c' if sig else '#95a5a6' for sig in ttest_df['Significant (p<0.05)']]
ax.barh(ttest_df['Feature'], -np.log10(ttest_df['P-Value']), color=colors)
ax.axvline(x=-np.log10(0.05), color='black', linestyle='--', label='p=0.05 threshold')
ax.set_xlabel('-log10(P-Value)')
ax.set_title('T-Test Significance (Continuous Features)')
ax.legend()

# Effect sizes
ax = axes[1]
colors = ['#e74c3c' if abs(d) > 0.5 else '#f39c12' if abs(d) > 0.2 else '#95a5a6' 
           for d in ttest_df["Cohen's d"]]
ax.barh(ttest_df['Feature'], ttest_df["Cohen's d"], color=colors)
ax.axvline(x=0.2, color='orange', linestyle='--', alpha=0.7, label='Small effect')
ax.axvline(x=-0.2, color='orange', linestyle='--', alpha=0.7)
ax.axvline(x=0.5, color='red', linestyle='--', alpha=0.7, label='Medium effect')
ax.axvline(x=-0.5, color='red', linestyle='--', alpha=0.7)
ax.set_xlabel("Cohen's d (Effect Size)")
ax.set_title('Effect Sizes (Continuous Features)')
ax.legend()

plt.tight_layout()
plt.savefig('ttest_significance.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Chi-Square Tests (Binary/Categorical Features)

Testing whether categorical features are independent of diagnosis.

- **H₀**: Feature and Diagnosis are independent
- **H₁**: Feature and Diagnosis are associated
- **α = 0.05**

In [ ]:
# Chi-square tests for binary/categorical features
chi2_results = []

for feature in binary_features:
    contingency = pd.crosstab(df_analysis[feature], df_analysis['Diagnosis'])
    chi2, p_value, dof, expected = chi2_contingency(contingency)
    
    # Cramér's V for effect size
    n = len(df_analysis)
    min_dim = min(contingency.shape) - 1
    cramers_v = np.sqrt(chi2 / (n * min_dim)) if min_dim > 0 else 0
    
    chi2_results.append({
        'Feature': feature,
        'Chi-Square': chi2,
        'P-Value': p_value,
        'Degrees of Freedom': dof,
        "Cramér's V": cramers_v,
        'Significant (p<0.05)': p_value < 0.05
    })

chi2_df = pd.DataFrame(chi2_results).sort_values('P-Value')
print("Chi-Square Test Results (sorted by p-value):")
print("=" * 80)
chi2_df

In [ ]:
# Visualize chi-square results
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

ax = axes[0]
colors = ['#e74c3c' if sig else '#95a5a6' for sig in chi2_df['Significant (p<0.05)']]
ax.barh(chi2_df['Feature'], -np.log10(chi2_df['P-Value'].clip(lower=1e-300)), color=colors)
ax.axvline(x=-np.log10(0.05), color='black', linestyle='--', label='p=0.05 threshold')
ax.set_xlabel('-log10(P-Value)')
ax.set_title('Chi-Square Significance (Categorical Features)')
ax.legend()

ax = axes[1]
colors = ['#e74c3c' if v > 0.3 else '#f39c12' if v > 0.1 else '#95a5a6' 
           for v in chi2_df["Cramér's V"]]
ax.barh(chi2_df['Feature'], chi2_df["Cramér's V"], color=colors)
ax.axvline(x=0.1, color='orange', linestyle='--', alpha=0.7, label='Small effect')
ax.axvline(x=0.3, color='red', linestyle='--', alpha=0.7, label='Medium effect')
ax.set_xlabel("Cramér's V (Effect Size)")
ax.set_title('Effect Sizes (Categorical Features)')
ax.legend()

plt.tight_layout()
plt.savefig('chi2_significance.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Mann-Whitney U Tests (Non-Parametric)

Non-parametric alternative to t-tests, does not assume normality.

In [ ]:
# Mann-Whitney U tests for all numeric features
all_features = [col for col in df_analysis.columns if col != 'Diagnosis']

mwu_results = []
for feature in all_features:
    group1 = diagnosed[feature].dropna()
    group2 = not_diagnosed[feature].dropna()
    
    u_stat, p_value = mannwhitneyu(group1, group2, alternative='two-sided')
    
    # Rank-biserial correlation (effect size for Mann-Whitney)
    n1, n2 = len(group1), len(group2)
    r = 1 - (2*u_stat) / (n1*n2)
    
    mwu_results.append({
        'Feature': feature,
        'U-Statistic': u_stat,
        'P-Value': p_value,
        'Rank-Biserial r': r,
        'Significant (p<0.05)': p_value < 0.05
    })

mwu_df = pd.DataFrame(mwu_results).sort_values('P-Value')
print("Mann-Whitney U Test Results (Top 15 most significant):")
print("=" * 80)
mwu_df.head(15)

## 4. Bonferroni Correction for Multiple Comparisons

When running many tests simultaneously, we need to correct for multiple comparisons to avoid false positives.

In [ ]:
# Combine all test results and apply Bonferroni correction
n_tests = len(all_features)
bonferroni_alpha = 0.05 / n_tests

print(f"Number of tests: {n_tests}")
print(f"Bonferroni-corrected alpha: {bonferroni_alpha:.6f}")
print(f"\n{'='*80}")
print(f"Features significant AFTER Bonferroni correction (Mann-Whitney U):")
print(f"{'='*80}")

mwu_df['Bonferroni Significant'] = mwu_df['P-Value'] < bonferroni_alpha
bonferroni_significant = mwu_df[mwu_df['Bonferroni Significant']].sort_values('P-Value')

if len(bonferroni_significant) > 0:
    print(f"\n{len(bonferroni_significant)} features remain significant after correction:\n")
    print(bonferroni_significant[['Feature', 'P-Value', 'Rank-Biserial r']].to_string(index=False))
else:
    print("\nNo features remain significant after Bonferroni correction.")
    print("Showing features significant at uncorrected p<0.05:")
    print(mwu_df[mwu_df['Significant (p<0.05)']].to_string(index=False))

## 5. Logistic Regression with P-Values

Using statsmodels to get coefficient significance for each predictor in a multivariate model.

In [ ]:
# Prepare features for logistic regression
X = df_analysis.drop(columns=['Diagnosis'])
y = df_analysis['Diagnosis']

# Standardize features
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

# Add constant for statsmodels
X_const = sm.add_constant(X_scaled)

# Fit logistic regression
logit_model = sm.Logit(y, X_const)
result = logit_model.fit(disp=0, maxiter=1000)

print(result.summary2())

In [ ]:
# Extract and display significant predictors
logit_summary = pd.DataFrame({
    'Feature': result.params.index,
    'Coefficient': result.params.values,
    'Std Error': result.bse.values,
    'Z-Score': result.tvalues.values,
    'P-Value': result.pvalues.values,
    'Odds Ratio': np.exp(result.params.values)
})

# Remove intercept for feature analysis
logit_features = logit_summary[logit_summary['Feature'] != 'const'].sort_values('P-Value')
logit_features['Significant (p<0.05)'] = logit_features['P-Value'] < 0.05

print("Logistic Regression - Significant Predictors (p < 0.05):")
print("=" * 80)
sig_predictors = logit_features[logit_features['Significant (p<0.05)']]
print(f"\n{len(sig_predictors)} significant predictors found:\n")
sig_predictors[['Feature', 'Coefficient', 'Odds Ratio', 'P-Value']]

In [ ]:
# Visualize logistic regression coefficients
fig, ax = plt.subplots(figsize=(10, 8))

logit_sorted = logit_features.sort_values('Coefficient')
colors = ['#e74c3c' if p < 0.05 else '#95a5a6' for p in logit_sorted['P-Value']]

ax.barh(logit_sorted['Feature'], logit_sorted['Coefficient'], color=colors)
ax.axvline(x=0, color='black', linewidth=0.5)
ax.set_xlabel('Standardized Coefficient')
ax.set_title('Logistic Regression Coefficients\n(Red = Significant at p<0.05)')

plt.tight_layout()
plt.savefig('logistic_regression_coefficients.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Random Forest Feature Importance

Using Random Forest to identify the most important features for predicting Alzheimer's diagnosis.

In [ ]:
# Train Random Forest
rf = RandomForestClassifier(n_estimators=500, random_state=42, n_jobs=-1)
rf.fit(X_scaled, y)

# Get feature importances
importance_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf.feature_importances_
}).sort_values('Importance', ascending=False)

print("Random Forest Feature Importance (Top 15):")
print("=" * 50)
importance_df.head(15)

In [ ]:
# Visualize feature importance
fig, ax = plt.subplots(figsize=(10, 8))

top_15 = importance_df.head(15).sort_values('Importance')
colors = plt.cm.RdYlGn(np.linspace(0.2, 0.9, len(top_15)))

ax.barh(top_15['Feature'], top_15['Importance'], color=colors)
ax.set_xlabel('Feature Importance (Gini)')
ax.set_title('Top 15 Features by Random Forest Importance')

plt.tight_layout()
plt.savefig('rf_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Combined Significance Summary

Combining results from all tests to identify consistently significant features.

In [ ]:
# Create a combined summary
summary = pd.DataFrame({'Feature': all_features})

# Merge Mann-Whitney results
summary = summary.merge(
    mwu_df[['Feature', 'P-Value', 'Significant (p<0.05)', 'Bonferroni Significant']].rename(
        columns={'P-Value': 'MWU P-Value', 
                 'Significant (p<0.05)': 'MWU Significant',
                 'Bonferroni Significant': 'MWU Bonferroni Sig'}),
    on='Feature', how='left'
)

# Merge logistic regression results
summary = summary.merge(
    logit_features[['Feature', 'P-Value', 'Significant (p<0.05)', 'Odds Ratio']].rename(
        columns={'P-Value': 'Logit P-Value',
                 'Significant (p<0.05)': 'Logit Significant'}),
    on='Feature', how='left'
)

# Merge RF importance
summary = summary.merge(
    importance_df.rename(columns={'Importance': 'RF Importance'}),
    on='Feature', how='left'
)

# Count how many tests found the feature significant
summary['Significance Score'] = (
    summary['MWU Significant'].astype(int) + 
    summary['Logit Significant'].astype(int) +
    (summary['RF Importance'] > summary['RF Importance'].median()).astype(int)
)

summary_sorted = summary.sort_values('Significance Score', ascending=False)
print("Combined Significance Summary:")
print("=" * 100)
print("(Score = number of methods agreeing the feature is significant)\n")
summary_sorted[['Feature', 'MWU P-Value', 'MWU Bonferroni Sig', 
                 'Logit P-Value', 'Logit Significant', 
                 'RF Importance', 'Significance Score']]

In [ ]:
# Final visualization: features significant across multiple tests
fig, ax = plt.subplots(figsize=(12, 8))

plot_data = summary_sorted.head(20).sort_values('Significance Score')
colors = ['#27ae60' if s == 3 else '#f39c12' if s == 2 else '#e74c3c' if s == 1 else '#95a5a6'
           for s in plot_data['Significance Score']]

ax.barh(plot_data['Feature'], plot_data['Significance Score'], color=colors)
ax.set_xlabel('Significance Score (out of 3 methods)')
ax.set_title('Feature Significance Across Multiple Statistical Methods\n'
             '(Green=3/3, Yellow=2/3, Red=1/3, Gray=0/3)')
ax.set_xticks([0, 1, 2, 3])

plt.tight_layout()
plt.savefig('combined_significance.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Distribution comparison for top significant features
top_sig_features = summary_sorted[summary_sorted['Significance Score'] == 3]['Feature'].tolist()[:6]

if len(top_sig_features) > 0:
    n_cols = 3
    n_rows = (len(top_sig_features) + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4*n_rows))
    axes = axes.flatten() if n_rows > 1 else [axes] if len(top_sig_features) == 1 else axes
    
    for idx, feature in enumerate(top_sig_features):
        ax = axes[idx]
        ax.hist(not_diagnosed[feature], bins=20, alpha=0.6, label='No Alzheimer\'s', color='#3498db')
        ax.hist(diagnosed[feature], bins=20, alpha=0.6, label='Alzheimer\'s', color='#e74c3c')
        ax.set_title(f'{feature}\n(Significant across all methods)')
        ax.legend()
    
    # Hide unused subplots
    for idx in range(len(top_sig_features), len(axes)):
        axes[idx].set_visible(False)
    
    plt.tight_layout()
    plt.savefig('top_significant_distributions.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("No features were significant across all 3 methods.")
    print("Showing top features by significance score instead.")
    print(summary_sorted[['Feature', 'Significance Score']].head(10))

## 8. Key Findings Summary

In [ ]:
print("\n" + "=" * 80)
print("SIGNIFICANCE ANALYSIS SUMMARY")
print("=" * 80)

# T-test significant features
ttest_sig = ttest_df[ttest_df['Significant (p<0.05)']]
print(f"\n1. T-TEST: {len(ttest_sig)}/{len(continuous_features)} continuous features significant")
if len(ttest_sig) > 0:
    for _, row in ttest_sig.iterrows():
        print(f"   - {row['Feature']}: p={row['P-Value']:.2e}, d={row["Cohen's d"]:.3f}")

# Chi-square significant features
chi2_sig = chi2_df[chi2_df['Significant (p<0.05)']]
print(f"\n2. CHI-SQUARE: {len(chi2_sig)}/{len(binary_features)} categorical features significant")
if len(chi2_sig) > 0:
    for _, row in chi2_sig.iterrows():
        print(f"   - {row['Feature']}: p={row['P-Value']:.2e}, V={row["Cramér's V"]:.3f}")

# Bonferroni-corrected significant features
bonf_sig = mwu_df[mwu_df['Bonferroni Significant']]
print(f"\n3. BONFERRONI-CORRECTED: {len(bonf_sig)}/{len(all_features)} features survive correction")
if len(bonf_sig) > 0:
    for _, row in bonf_sig.iterrows():
        print(f"   - {row['Feature']}: p={row['P-Value']:.2e}")

# Logistic regression significant features
logit_sig = logit_features[logit_features['Significant (p<0.05)']]
print(f"\n4. LOGISTIC REGRESSION: {len(logit_sig)}/{len(all_features)} features significant")
if len(logit_sig) > 0:
    for _, row in logit_sig.iterrows():
        direction = 'increases' if row['Odds Ratio'] > 1 else 'decreases'
        print(f"   - {row['Feature']}: OR={row['Odds Ratio']:.3f} ({direction} risk), p={row['P-Value']:.2e}")

# Top RF features
print(f"\n5. RANDOM FOREST TOP 5 FEATURES:")
for _, row in importance_df.head(5).iterrows():
    print(f"   - {row['Feature']}: importance={row['Importance']:.4f}")

# Overall most significant
most_sig = summary_sorted[summary_sorted['Significance Score'] >= 2]['Feature'].tolist()
print(f"\n{'='*80}")
print(f"MOST SIGNIFICANT FEATURES (agreed upon by 2+ methods):")
print(f"{'='*80}")
for f in most_sig:
    score = summary_sorted[summary_sorted['Feature'] == f]['Significance Score'].values[0]
    print(f"   ★ {f} (score: {score}/3)")